# Funciones como valores: lambda, map, filter, sorted, any, all

En Python las funciones son objetos como cualquier otro: se pueden guardar en
variables, meter en listas o diccionarios y pasar como argumento a otras funciones.
Esta idea es la base de todo lo que vamos a ver hoy:

- Funciones como variables
- Tablas de despacho (dispatch tables)
- Funciones `lambda`
- `map`
- `filter`
- Iteradores perezosos
- `any` y `all`
- `sorted`, `min` y `max` con `key`


## Datos de ejemplo

Usamos un diccionario con la misma forma que el del proyecto: clave = ciudad,
valor = diccionario con el resto de los campos. El viento ya está separado en
dirección y velocidad, y la sensación térmica es `None` cuando el SMN informa
"No se calcula".

In [1]:
datos = {
    "Bariloche":     {"temperatura": 4.2,  "humedad": 81, "viento_dir": "Oeste", "viento_vel": 22, "sensacion": -0.5},
    "Buenos Aires":  {"temperatura": 17.8, "humedad": 64, "viento_dir": "Este",  "viento_vel": 13, "sensacion": None},
    "Córdoba":       {"temperatura": 21.3, "humedad": 45, "viento_dir": "Norte", "viento_vel": 9,  "sensacion": None},
    "Esquel":        {"temperatura": 1.6,  "humedad": 88, "viento_dir": "Oeste", "viento_vel": 31, "sensacion": -4.8},
    "Mendoza":       {"temperatura": 19.4, "humedad": None, "viento_dir": "Sur", "viento_vel": 5,  "sensacion": None},
    "Ushuaia":       {"temperatura": -1.2, "humedad": 90, "viento_dir": "Sudoeste", "viento_vel": 40, "sensacion": -8.3},
    "Salta":         {"temperatura": 23.0, "humedad": 38, "viento_dir": "Calma", "viento_vel": 0,  "sensacion": None},
}
temperaturas = [d["temperatura"] for d in datos.values()]
temperaturas

[4.2, 17.8, 21.3, 1.6, 19.4, -1.2, 23.0]

## Funciones como variables

El nombre de una función es una variable que apunta a un objeto función.
Sin paréntesis hablamos de la función; con paréntesis la llamamos.

In [2]:
def c_a_f(celsius):
    """Convierte grados Celsius a Fahrenheit."""
    return celsius * 9 / 5 + 32

print(c_a_f)        # el objeto función
print(c_a_f(20))    # el resultado de llamarla
print(type(c_a_f))

<function c_a_f at 0x7c5784eb73d0>
68.0
<class 'function'>


In [3]:
convertir = c_a_f      # otra variable que apunta a la MISMA función (no se llama)
convertir(100)

212.0

Como es un valor, una función puede ser argumento de otra función.

In [ ]:
def aplicar_a_todos(funcion, valores):
    """Devuelve una lista nueva con funcion aplicada a cada valor."""
    resultado = []
    for v in valores:
        resultado.append(funcion(v))
    return resultado

aplicar_a_todos(c_a_f, temperaturas)

In [ ]:
aplicar_a_todos(round, temperaturas)   # funciones de Python también sirven

## Tablas de despacho (dispatch tables)

Un diccionario cuyos valores son funciones. Sirve para elegir qué hacer a partir
de un texto u otra clave, sin una cadena larga de `if / elif`.

In [ ]:
# Versión con if/elif
def resumir_if(valores, operacion):
    if operacion == "max":
        return max(valores)
    elif operacion == "min":
        return min(valores)
    elif operacion == "suma":
        return sum(valores)
    else:
        raise ValueError(f"Operación desconocida: {operacion}")

resumir_if(temperaturas, "max")

In [ ]:
def promedio(valores):
    return sum(valores) / len(valores)

# Versión con tabla de despacho
OPERACIONES = {
    "max": max,
    "min": min,
    "suma": sum,
    "promedio": promedio,   # también funciones nuestras
}

def resumir(valores, operacion):
    if operacion not in OPERACIONES:
        raise ValueError(f"Operación desconocida: {operacion}")
    return OPERACIONES[operacion](valores)

for op in OPERACIONES:
    print(op, "->", resumir(temperaturas, op))

Agregar una operación nueva es agregar una línea al diccionario: la función
`resumir` no cambia. Es útil, por ejemplo, para elegir qué hacer según un argumento
de línea de comandos (`sys.argv`).

## Funciones `lambda`

Una `lambda` es una función chica, sin nombre, de una sola expresión
(sin `return`: el valor de la expresión es lo que devuelve).

```python
lambda parametros: expresion
```

In [9]:
def al_cuadrado(x):
    return x ** 2

al_cuadrado_lambda = lambda x: x ** 2   # equivalente (solo para mostrar)

print(al_cuadrado(5), al_cuadrado_lambda(5))

25 25


In [8]:
# cortando líneas
listita = [1, 2, 3, 4,
          5, 6, 7, 9]

> Asignar una `lambda` a un nombre, como arriba, no se recomienda: para eso está `def`.
> Las `lambda` brillan cuando se escriben directamente como argumento de otra función.

In [ ]:
aplicar_a_todos(lambda t: round(t + 273.15, 2), temperaturas)   # a Kelvin, sin definir nada aparte

In [ ]:
# Puede tener varios parámetros
diferencia = lambda a, b: abs(a - b)
diferencia(4.2, -1.2)

## `map`

`map(funcion, iterable)` aplica la función a cada elemento. Es lo que hicimos a mano
con `aplicar_a_todos`, pero ya viene en Python.

In [10]:
temperaturas

[4.2, 17.8, 21.3, 1.6, 19.4, -1.2, 23.0]

In [19]:
en_fahrenheit = map(c_a_f, temperaturas)
en_fahrenheit

In [15]:
import sys
sys.getsizeof(temperaturas)

120

In [16]:
sys.getsizeof(en_fahrenheit)

56

In [20]:
list(en_fahrenheit)       # list(...) para ver los resultados

[39.56, 64.04, 70.34, 34.88, 66.92, 29.84, 73.4]

In [21]:
list(en_fahrenheit)

[]

In [22]:
# Ejemplo del proyecto: limpiar espacios de los nombres de ciudad
nombres_crudos = ["  Bariloche", "Esquel ", " Ushuaia  "]
list(map(str.strip, nombres_crudos))

['Bariloche', 'Esquel', 'Ushuaia']

In [ ]:
# Convertir textos leídos de un archivo a números
campos = ["4.2", "17.8", "21.3"]
list(map(float, campos))

In [ ]:
# Con lambda
list(map(lambda t: round(t), temperaturas))

### `map` con funciones de más de un parámetro

Hay dos situaciones distintas:

a) Cada parámetro viene de su propia lista. Se le pasan a `map` varios
iterables: toma un elemento de cada uno, en paralelo.

In [ ]:
def sensacion_por_viento(temperatura, viento):
    """Fórmula de wind chill (Environment Canada), válida para T <= 10 °C y viento > 4.8 km/h."""
    v = viento ** 0.16
    return round(13.12 + 0.6215 * temperatura - 11.37 * v + 0.3965 * temperatura * v, 1)

temps  = [4.2, 1.6, -1.2]
vientos = [22, 31, 40]

list(map(sensacion_por_viento, temps, vientos))   # (4.2, 22), (1.6, 31), (-1.2, 40)

In [ ]:
# También con lambda de dos parámetros
list(map(lambda t, h: f"{t} °C / {h} %", [4.2, 17.8], [81, 64]))

In [ ]:
# Si las listas tienen distinto largo, map corta en la más corta
list(map(lambda a, b: a + b, [1, 2, 3, 4], [10, 20]))

b) Un parámetro varía y los otros quedan fijos. Envolvemos la llamada en una
`lambda` de un solo parámetro que completa los demás.

In [ ]:
def convertir(temperatura, unidad):
    if unidad == "F":
        return round(temperatura * 9 / 5 + 32, 2)
    elif unidad == "K":
        return round(temperatura + 273.15, 2)
    return temperatura

print(list(map(lambda t: convertir(t, "K"), temperaturas)))
print(list(map(lambda t: convertir(t, "F"), temperaturas)))

In [ ]:
# Lo mismo con una comprensión, para comparar
[convertir(t, "K") for t in temperaturas]

La alternativa idiomática: comprensión de listas. Hace lo mismo y en Python
suele leerse mejor, sobre todo cuando habría que escribir una `lambda`.

In [ ]:
print(list(map(lambda t: t * 9 / 5 + 32, temperaturas)))
print([t * 9 / 5 + 32 for t in temperaturas])

## `filter`

`filter(funcion, iterable)` se queda con los elementos para los que la función
devuelve algo verdadero.

In [ ]:
bajo_cero = filter(lambda t: t < 0, temperaturas)
list(bajo_cero)

In [ ]:
# Sobre un diccionario: ciudades con viento fuerte (> 20 km/h)
list(filter(lambda ciudad: datos[ciudad]["viento_vel"] > 20, datos))

In [ ]:
# Con una función con nombre, cuando la condición es más larga
def es_fria_y_ventosa(ciudad):
    d = datos[ciudad]
    return d["temperatura"] < 5 and d["viento_vel"] > 20

list(filter(es_fria_y_ventosa, datos))

In [ ]:
# Lo mismo con comprensión
[c for c in datos if datos[c]["temperatura"] < 5 and datos[c]["viento_vel"] > 20]

In [ ]:
# map + filter combinados vs comprensión
print(list(map(c_a_f, filter(lambda t: t < 5, temperaturas))))
print([c_a_f(t) for t in temperaturas if t < 5])

## Iteradores perezosos (lazy)

`map` y `filter` no devuelven una lista. Devuelven un objeto que calcula cada
elemento recién cuando alguien lo pide. Eso tiene dos consecuencias:

1. Si los imprimimos directamente no vemos los datos.
2. Se consumen una sola vez.

In [ ]:
m = map(c_a_f, temperaturas)
print(m)          # todavía no calculó nada
print(list(m))    # ahora sí
print(list(m))    # ¡vacío! ya se consumió

Para ver cuándo se calcula cada valor, usamos una función que avisa al ejecutarse.

In [ ]:
def c_a_f_ruidosa(t):
    print(f"  calculando {t}...")
    return c_a_f(t)

m = map(c_a_f_ruidosa, temperaturas)
print("Creé el map. ¿Se calculó algo? No.")

In [ ]:
print("Pido el primero con next():")
print(next(m))
print("Pido el segundo:")
print(next(m))

In [ ]:
print("Recorro el resto con un for:")
for f in m:
    print(f)

Si necesitamos usar el resultado más de una vez, lo convertimos a lista:

In [ ]:
fahrenheit = list(map(c_a_f, temperaturas))
print(max(fahrenheit), min(fahrenheit))   # la lista se puede recorrer las veces que quieras

### Generadores

Una comprensión con paréntesis en vez de corchetes es un generador:
la versión perezosa de la comprensión de listas.

In [ ]:
lista = [t * 2 for t in temperaturas]   # se calcula todo ya, ocupa memoria
gen   = (t * 2 for t in temperaturas)   # perezoso, como map

print(lista)
print(gen)
print(sum(gen))   # se puede pasar directo a sum, max, any, all...

## `any` y `all`

- `any(iterable)` → `True` si al menos un elemento es verdadero.
- `all(iterable)` → `True` si todos son verdaderos.

In [ ]:
print(any(t < 0 for t in temperaturas))     # ¿hay alguna bajo cero?
print(all(t > -10 for t in temperaturas))   # ¿son todas mayores a -10?
print(all(t > 0 for t in temperaturas))     # ¿son todas positivas?

### Ejemplo del proyecto: ciudades con todos los datos completos

In [ ]:
def esta_completa(registro):
    return all(valor is not None for valor in registro.values())

print(esta_completa(datos["Bariloche"]))
print(esta_completa(datos["Mendoza"]))

In [ ]:
completas = [c for c in datos if esta_completa(datos[c])]
print(completas)
print("Cantidad:", len(completas))

### Cortocircuito

`any` se detiene en el primer verdadero y `all` en el primer falso:
no revisan el resto. Es la pereza en acción.

In [ ]:
def bajo_cero_ruidosa(t):
    print(f"  revisando {t}")
    return t < 0

print(any(bajo_cero_ruidosa(t) for t in [5, 3, -1, 8, 10]))

In [ ]:
print(all(bajo_cero_ruidosa(t) for t in [-2, 4, -1, -8]))

### Detalles a tener en cuenta

In [ ]:
print(any([]))   # False: no hay ninguno verdadero
print(all([]))   # True: no hay ninguno que lo contradiga

In [ ]:
# Cuidado con la "veracidad": 0 y "" cuentan como falsos.
# Un viento de 0 km/h (calma) es un dato válido, no un dato faltante.
registro = {"temperatura": 23.0, "viento_dir": "Calma", "viento_vel": 0}
print(all(registro.values()))                            # False: engaña
print(all(v is not None for v in registro.values()))     # True: correcto

## `sorted`, `min` y `max` con `key`

`sorted` ordena cualquier iterable y devuelve una lista nueva. Con `key` le
decimos según qué ordenar: una función que se aplica a cada elemento y cuyo
resultado se usa para comparar.

In [ ]:
print(sorted(temperaturas))
print(sorted(temperaturas, reverse=True))

In [ ]:
nombres = ["esquel", "Ushuaia", "bariloche", "Córdoba"]
print(sorted(nombres))                  # las mayúsculas van antes que las minúsculas
print(sorted(nombres, key=str.lower))   # ordena sin distinguir mayúsculas
print(sorted(nombres, key=len))         # por largo del nombre

### `sorted` vs `.sort()`

In [ ]:
valores = [3, 1, 2]
nueva = sorted(valores)
print(valores, nueva)        # sorted no modifica la original

valores.sort()               # .sort() modifica la lista en el lugar...
print(valores)

resultado = valores.sort()   # ...y devuelve None
print(resultado)             # error clásico: lista = lista.sort()

### Ordenar un diccionario

Recorrer un diccionario da sus claves. Para ordenar ciudades según un campo,
la `key` recibe una ciudad y devuelve el valor a comparar.

In [ ]:
sorted(datos, key=lambda c: datos[c]["temperatura"])

Con `.items()` obtenemos pares `(ciudad, registro)`; la `key` recibe el par.

In [ ]:
por_viento = sorted(datos.items(), key=lambda par: par[1]["viento_vel"], reverse=True)
for ciudad, registro in por_viento:
    print(f"{ciudad:<14} {registro['viento_vel']:>3} km/h")

### Varios criterios: `key` que devuelve una tupla

Las tuplas se comparan elemento por elemento: si el primero empata, decide el segundo.

In [ ]:
# Por dirección del viento y, dentro de cada dirección, por velocidad
for c in sorted(datos, key=lambda c: (datos[c]["viento_dir"], datos[c]["viento_vel"])):
    print(f"{datos[c]['viento_dir']:<9} {datos[c]['viento_vel']:>3}  {c}")

El ordenamiento es estable: si dos elementos empatan, conservan el orden que
tenían. Por eso se puede ordenar en dos pasadas (primero por el criterio secundario).

### `min` y `max` también aceptan `key`

Cuando solo queremos el extremo no hace falta ordenar todo.

In [ ]:
mas_fria = min(datos, key=lambda c: datos[c]["temperatura"])
mas_ventosa = max(datos, key=lambda c: datos[c]["viento_vel"])
print(mas_fria, datos[mas_fria]["temperatura"])
print(mas_ventosa, datos[mas_ventosa]["viento_vel"])

## Integrador: las n ciudades extremas

El enunciado pide listar las n ciudades más cálidas/frías y con más/menos viento,
con una función reutilizable para ambos campos. Pasando el campo como parámetro,
sale con `sorted` y `key`.

In [ ]:
def extremos(datos, campo, n, mayores=True):
    """Devuelve las n ciudades con mayor (o menor) valor en campo.

    Las ciudades sin dato en ese campo (None) se descartan.
    """
    con_dato = filter(lambda c: datos[c][campo] is not None, datos)
    ordenadas = sorted(con_dato, key=lambda c: datos[c][campo], reverse=mayores)
    return ordenadas[:n]

print("Más cálidas:  ", extremos(datos, "temperatura", 3))
print("Más frías:    ", extremos(datos, "temperatura", 3, mayores=False))
print("Más viento:   ", extremos(datos, "viento_vel", 2))
print("Más húmedas:  ", extremos(datos, "humedad", 2))     # Mendoza no tiene dato

Y con una tabla de despacho podemos armar un pequeño menú de reportes:

In [ ]:
REPORTES = {
    "calidas":  lambda: extremos(datos, "temperatura", 3),
    "frias":    lambda: extremos(datos, "temperatura", 3, mayores=False),
    "ventosas": lambda: extremos(datos, "viento_vel", 3),
    "completas": lambda: [c for c in datos if esta_completa(datos[c])],
}

for nombre, reporte in REPORTES.items():
    print(f"{nombre:<10} {reporte()}")